In [0]:
df_category = spark.read.table("ecommerce.bronze.brz_category")


In [0]:
from pyspark.sql.functions import col
df_category.select(col("category_code")).distinct().show()

In [0]:
# Check if we have duplicates
df_category.groupBy("category_code").count().filter(col("count") > 1).show()

df_category = df_category.dropDuplicates(["category_code"])

df_category.groupBy("category_code").count().show()

### Alternative (just for KNowledge)

In [0]:
dd = spark.sql("select category_code, count(*) from ecommerce.bronze.brz_category group by category_code")
dd.show()

### Convert Catwgory to Uppercase 

In [0]:
from pyspark.sql.functions import col, upper
df_category = df_category.withColumn("category_code", upper(col("category_code")))
df_category.show()

### Write this Silver version to s3 and also save as table in Silver schema 

In [0]:
df_category.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("s3://sj-dbr-demo-proj/silver_data/slv_category")

# Register table so we have LHS table 
spark.sql("""
          create table if not exists ecommerce.silver.slv_category
          using delta
          location 's3://sj-dbr-demo-proj/silver_data/slv_category'
          """)

In [0]:
dbutils.notebook.exit("SUCCESS")